In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import os

# Directory containing your CSVs
csv_dir = "specify"

# Load each CSV with the correct quote handling
dfs = []
for split in ["train", "val", "test"]:
    csv_path = os.path.join(csv_dir, f"{split}.csv")
    df = pd.read_csv(csv_path, quotechar='"')
    dfs.append(df)

# Combine into one DataFrame
df = pd.concat(dfs, ignore_index=True)

# Preview
df

,text,label,split
0,The Flaming Lips Plus Special Guests Thu 9 Nov...,a38_ticket,train
1,Standing Theatre 25p Olivier Circle Aislel EVE...,a38_ticket,train
2,GREEN SIDE LEKEL GREATER LONDON COUNCIL TUESDA...,a38_ticket,train
3,UNITEDKINGDOMOFGREATBRITAINANDNORTHERNIRELAND ...,a31_passport,train
4,A TUTTA BIRRA? SCEQLIIL SUPERMENU SUMMERI - dl...,a37_receipt,train
...,...,...,...
1953,UNIONE EUROPEA REPUBBLICA ITALIANA PASSAPORTO,a31_passport,test
1954,CENTERFOR MEDIA HEPALEY PALEYFEST20M &GEEKSUND...,a38_ticket,test
1955,aanan YmtMayV1935 VearnfMom Ymedo ta iuhres me...,a35_mail,test
1956,14.pononHMT EOUOH.PVHAPA.KUEPZBOWENRICHARD_CQO...,a31_passport,test


In [ ]:
df["label"].value_counts()

,count
label,
a38_ticket,907
a31_passport,285
a37_receipt,260
a30_credit_card,175
a35_mail,173
a33_student_id,82
a32_drivers_license,76


In [ ]:
import pandas as pd

# Shuffle and sample 400 rows from train split
train_subset = df[df["split"] == "train"].sample(n=400, random_state=42)

# Split into 200 for val and 200 for test
val_split = train_subset.iloc[:200].copy()
test_split = train_subset.iloc[200:].copy()

# Update split labels
val_split["split"] = "val"
test_split["split"] = "test"

# Remove sampled rows from training set
df = df.drop(train_subset.index)

# Add to val and test sets
df = pd.concat([df, val_split, test_split], ignore_index=True)

print("[INFO] 400 rows moved: 200 to validation, 200 to test")


[INFO] 400 rows moved: 200 to validation, 200 to test


In [ ]:
df["split"].value_counts()

,count
split,
train,1083
val,450
test,425


In [ ]:
!pip install transformers datasets


In [ ]:
import random
import pandas as pd

def inject_ocr_noise(text, error_rate=0.05):
    chars = list(text)
    i = 0
    while i < len(chars):
        if random.random() < error_rate:
            op = random.choice(["swap", "delete", "repeat"])
            if op == "swap" and i < len(chars) - 1:
                chars[i], chars[i+1] = chars[i+1], chars[i]
                i += 1
            elif op == "delete":
                del chars[i]
                continue
            elif op == "repeat":
                chars.insert(i, chars[i])
                i += 1
        i += 1
    return ''.join(chars)

def augment_classes(df, min_threshold, max_threshold, n_augments):
    class_counts = df["label"].value_counts()
    target_classes = class_counts[
        (class_counts >= min_threshold) & (class_counts < max_threshold)
    ].index.tolist()

    print(f"[INFO] Augmenting {len(target_classes)} classes between {min_threshold} and {max_threshold} samples")

    augmented_rows = []

    for label in target_classes:
        subset = df[(df["label"] == label) & (df["split"] == "train")]
        for _, row in subset.iterrows():
            for _ in range(n_augments):
                noisy_text = inject_ocr_noise(row["text"])
                augmented_rows.append({
                    "text": noisy_text,
                    "label": label,
                    "split": row["split"]
                })

    df_augmented = pd.DataFrame(augmented_rows)
    return pd.concat([df, df_augmented], ignore_index=True)


In [ ]:
df = augment_classes(df, 300, 1000, 1)
df = augment_classes(df, 200, 300, 3)
df = augment_classes(df, 100, 200, 4)
df = augment_classes(df, 0, 100, 5)

[INFO] Augmenting 1 classes between 300 and 1000 samples
[INFO] Augmenting 2 classes between 200 and 300 samples
[INFO] Augmenting 2 classes between 100 and 200 samples
[INFO] Augmenting 2 classes between 0 and 100 samples


In [ ]:
df["label"].value_counts()

,count
label,
a38_ticket,1402
a31_passport,783
a37_receipt,686
a35_mail,577
a30_credit_card,559
a33_student_id,302
a32_drivers_license,271


In [ ]:
df["split"].value_counts()

,count
split,
train,3705
val,450
test,425


In [ ]:
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder

# Label encode
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])

# Save label mapping for later
label2id = {label: idx for idx, label in enumerate(le.classes_)}
id2label = {v: k for k, v in label2id.items()}

# Load tokenizer
model_name = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=256)

from datasets import Dataset
dataset = Dataset.from_pandas(df[["text", "label_id", "split"]])
tokenized = dataset.map(tokenize, batched=True)


/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/4580 [00:00<?, ? examples/s]

In [ ]:
train_ds = tokenized.filter(lambda x: x["split"] == "train")
val_ds = tokenized.filter(lambda x: x["split"] == "val")
test_ds = tokenized.filter(lambda x: x["split"] == "test")

# Remove the "split" column before training
train_ds = train_ds.remove_columns("split")
val_ds = val_ds.remove_columns("split")
test_ds = test_ds.remove_columns("split")


Filter:   0%|          | 0/4580 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4580 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4580 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import classification_report

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean()}

train_ds = train_ds.rename_column("label_id", "labels")
val_ds = val_ds.rename_column("label_id", "labels")
test_ds = test_ds.rename_column("label_id", "labels")


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.460300,1.053459,0.751111
2,0.328200,0.895308,0.824444
3,0.166400,1.041843,0.815556


TrainOutput(global_step=1392, training_loss=0.4730339372863768, metrics={'train_runtime': 382.6891, 'train_samples_per_second': 29.044, 'train_steps_per_second': 3.637, 'total_flos': 736279435906560.0, 'train_loss': 0.4730339372863768, 'epoch': 3.0})

In [ ]:
preds = trainer.predict(test_ds)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=le.classes_))


                     precision    recall  f1-score   support

    a30_credit_card       0.77      0.87      0.82        31
       a31_passport       0.85      0.86      0.86        66
a32_drivers_license       0.61      1.00      0.76        14
     a33_student_id       0.62      0.87      0.72        15
           a35_mail       0.69      0.65      0.67        34
        a37_receipt       0.77      0.67      0.72        55
         a38_ticket       0.93      0.88      0.90       210

           accuracy                           0.84       425
          macro avg       0.75      0.83      0.78       425
       weighted avg       0.84      0.84      0.84       425

